# Sunfish heavy jobs — Colab runner

Runs the CPU-heavy Sunfish tasks on Colab instead of a laptop:
full-scale calibration corpus assembly (75M tokens), tokenization,
and the repo test suite. **Runtime → Run all** after setting the two
options in the next cell. A free CPU runtime is fine; no GPU needed.

Outputs land in `/content/out` and are zipped at the end (optionally
copied to your Drive). The HF token is requested interactively and
never written to disk.

In [ ]:
TOKENS_PER_BUCKET = 12_500_000   # 6 buckets -> ~75M tokens (stage-1 full corpus)
COPY_TO_DRIVE = False            # True: mount Drive and copy the zip there


In [ ]:
%%bash
set -e
rm -rf /content/Sunfish
git clone --depth 1 https://github.com/Sculptor-AI/Sunfish /content/Sunfish
pip install --quiet tokenizers datasets huggingface_hub
mkdir -p /content/out


In [ ]:
from getpass import getpass
from huggingface_hub import hf_hub_download, login

login(token=getpass('HF token (license-accepted account): '))
tokenizer_path = hf_hub_download(
    'google/diffusiongemma-26B-A4B-it', 'tokenizer.json', local_dir='/content/out'
)
print('tokenizer at', tokenizer_path)


In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '/content/Sunfish/scripts/assemble_calibration.py',
     '--tokenizer', '/content/out/tokenizer.json',
     '--output', '/content/out/calibration',
     '--tokens-per-bucket', str(TOKENS_PER_BUCKET)],
    env={'PYTHONPATH': '/content/Sunfish/src', 'PATH': '/usr/bin:/bin:/usr/local/bin',
         'HOME': '/root'},
)
assert result.returncode == 0, 'assembly failed — read the log above'


In [ ]:
%%bash
cd /content/Sunfish && PYTHONPATH=src python3 -m unittest discover -s tests 2>&1 | tail -3


In [ ]:
import shutil

archive = shutil.make_archive('/content/sunfish-calibration', 'zip', '/content/out/calibration')
print('artifact:', archive)
if COPY_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copy(archive, '/content/drive/MyDrive/sunfish-calibration.zip')
    print('copied to Drive')
